<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# IMU Basics

A Duckiedrone must control its attitude to direct the thrust that supports and moves it. Its __inertial measurement unit (IMU)__ supplies acceleration and angular-velocity measurements that help the flight controller estimate that attitude and respond to rotation. IMUs also appear in devices ranging from smartphones to other types of robots; on the Duckiedrone, the tiny sensor chip is built into the flight-controller board shown in [Figure 1](#figure-1).

<figure id="figure-1" class="lx-figure" style="width:420px;">
  <img src="../assets/_images/fc-405v1-dd24-b-front.png" style="width:100%;" />
  <figcaption>
    Figure 1: The IMU is embedded in the flight-controller board. The silkscreen arrow marks the board's forward direction; the chip's sensing axes and their mounting rotation must be established separately. 
  </figcaption>
</figure>

An IMU is a sensor package that combines an accelerometer and a gyroscope (occasionally a magnetometer too). This notebook explains how these sensors produce measurements and how to interpret them, and uses a recorded message to verify typical measurements on a stationary Duckiedrone.

Integrating the measurements over time also contributes to estimates of attitude, velocity, and position, provided the initial conditions and coordinate frames are known. Measurement errors accumulate through integration, which limits the accuracy of these estimates. This notebook addresses the following questions:

- What does an IMU measure, and what does it not measure?

- How do the accelerometer and gyroscope produce their readings?

- How does proper acceleration differ from linear acceleration?

- Why does the `linear_acceleration` field of `sensor_msgs/Imu` not hold linear acceleration on the Duckiedrone?

## What does an IMU measure?

The accelerometer measures acceleration relative to free fall, called proper acceleration, while the gyroscope measures angular velocity, that is, how fast the sensor turns and about which axis. Some IMUs add a magnetometer, which measures the Earth's magnetic field and can provide a heading reference, but the Duckiedrone's IMU does not include one.

[Figure 2](#figure-2) shows the two sensors packaged inside the IMU. Each reports three components, corresponding to the sensor's $x$, $y$, and $z$ axes.

<figure id="figure-2" class="lx-figure" style="width:420px;">
  <img src="../assets/_images/imu.png" style="width:100%;" />
  <figcaption>
    Figure 2: The accelerometer measures proper acceleration along the sensor's three axes
    (straight arrows), while the gyroscope measures angular velocity about them
    (curved arrows). See the <a
    href="https://api.semanticscholar.org/CorpusID:6415932">image source</a>.
  </figcaption>
</figure>

[Table 1](#table-1) compares their outputs.

<a id="table-1"></a>

Table 1: Measurements provided by a typical two-sensor IMU.

| Sensor | What it measures | Units |
| --- | --- | --- |
| Accelerometer | Proper acceleration along sensor $x$, $y$, and $z$ axes | $\mathrm{m/s^2}$ |
| Gyroscope | Angular velocity about sensor $x$, $y$, and $z$ axes | $\mathrm{rad/s}$ |


### The accelerometer

A microelectromechanical system (MEMS) combines microscopic mechanical structures with electronics. In a capacitive MEMS accelerometer, flexible structures suspend a small mass inside the sensor and act as springs. When the mass moves relative to the housing, it changes the capacitance between moving and fixed electrodes, which the electronics convert into a proper-acceleration reading.

[Figure 3](#figure-3) represents this mechanism as a mass-spring-damper system in which the housing moves with the Duckiedrone and the suspended mass can move relative to it. As the mass moves, the spring provides a restoring force while the damper reduces oscillations.

<figure id="figure-3" class="lx-figure" style="width:640px;">
  <img src="../assets/_images/mass_spring_damper.png" style="width:100%;" />
  <figcaption>
    Figure 3: A simplified accelerometer model in which the displacement of the suspended mass relative to the housing is related to proper acceleration.
  </figcaption>
</figure>

Once the suspended mass has settled, its displacement is proportional to proper acceleration.

To relate this mechanism to motion, let $t$ denote time and use bold symbols for vectors, with $\mathbf{x}(t)$ representing the sensor's position relative to a stationary world frame. A dot denotes a time derivative, so linear velocity $\mathbf{v}(t)=\dot{\mathbf{x}}(t)$ is the rate of change of position and linear acceleration $\mathbf{a}(t)=\dot{\mathbf{v}}(t)$ is the rate of change of velocity. With $\mathbf{g}$ denoting gravitational acceleration, proper acceleration $\mathbf{f}(t)$ is:

<span id="equation-1"></span>

$$
\mathbf{f}(t) = \mathbf{a}(t) - \mathbf{g}.
\tag{1}
$$

Vector components must be expressed in the same frame before addition or subtraction, so subscripts identify the frame used in each equation. The sensor frame, whose axes are fixed to the sensor, will use the subscript $\mathrm{S}$:

<span id="equation-2"></span>

$$
\mathbf{f}_{\mathrm{S}}(t)
= \mathbf{a}_{\mathrm{S}}(t) - \mathbf{g}_{\mathrm{S}}(t).
\tag{2}
$$

Near the Earth's surface, gravitational acceleration points downward and has magnitude approximately $9.81\,\mathrm{m/s^2}$. For a level sensor whose positive $z$ axis points upward, $\mathbf{g}_{\mathrm{S}}=(0,0,-9.81)^{\mathsf{T}}\,\mathrm{m/s^2}$, where $\mathsf{T}$ (for "transpose") indicates that the three components form a column vector. Let $a_z$ and $f_z$ denote the vertical components of linear and proper acceleration in these axes. [Equation (2)](#equation-2) then gives:

$$
f_z = a_z + 9.81\,\mathrm{m/s^2}.
$$

At rest on a table, the housing is supported by the table while the spring supports the suspended mass against gravity. Since $a_z=0$, the ideal vertical reading is $f_z=+9.81\,\mathrm{m/s^2}$, which is also the expected reading for a Duckiedrone hovering motionless with support provided by its propellers.

If the level sensor accelerates upward at $a_z=+2\,\mathrm{m/s^2}$, the ideal vertical reading increases to $f_z=+11.81\,\mathrm{m/s^2}$. The reading therefore depends on acceleration, even when the sensor's orientation stays the same.

In ideal free fall, gravity is the only external force acting on the falling system, so $\mathbf{a}_{\mathrm{S}}=\mathbf{g}_{\mathrm{S}}$ and $\mathbf{f}_{\mathrm{S}}=\mathbf{0}$, where $\mathbf{0}$ denotes the zero vector. The housing and mass fall together, and the settled model has no spring deflection.

An ideal accelerometer reports $\mathbf{f}_{\mathrm{S}}(t)$ exactly, whereas a physical measurement can differ from this value. Write the measured quantity as $\tilde{\mathbf{f}}_{\mathrm{S}}(t)$, using a tilde to distinguish it from the ideal value, and model the difference as an offset, or bias, $\mathbf{b}_{a,\mathrm{S}}$, plus random measurement noise $\mathbf{n}_{a,\mathrm{S}}(t)$:

<span id="equation-3"></span>

$$
\tilde{\mathbf{f}}_{\mathrm{S}}(t)
= \mathbf{f}_{\mathrm{S}}(t)
+ \mathbf{b}_{a,\mathrm{S}}
+ \mathbf{n}_{a,\mathrm{S}}(t).
\tag{3}
$$

In [Equation (3)](#equation-3), bias is treated as constant over the interval being studied, while noise varies between samples; both terms have units of $\mathrm{m/s^2}$. Estimating their contributions requires a sequence of measurements so that a persistent offset can be distinguished from random variation.

### The gyroscope

A MEMS gyroscope keeps a small mass vibrating so that rotation about its sensing axis produces a detectable response. As the sensor rotates, the Coriolis effect produces a force on the vibrating mass perpendicular to its driven motion. Capacitive electrodes detect the resulting deflection, which the electronics use to determine the magnitude and sign of the angular velocity.

The output describes the rate of rotation, so recovering an angle requires accumulating that rate over time from a known starting orientation.

Intuitively, a person sitting in a rotating office chair can distinguish how fast the chair is turning from which direction it faces. Determining the final direction from the rate requires adding up the rotation from a known starting direction.

Let $\boldsymbol{\omega}_{\mathrm{S}}(t)$ denote angular velocity in sensor-frame components, measured in $\mathrm{rad/s}$. The measured vector $\tilde{\boldsymbol{\omega}}_{\mathrm{S}}(t)$ can be modeled using a constant bias $\mathbf{b}_{\omega,\mathrm{S}}$ and random noise $\mathbf{n}_{\omega,\mathrm{S}}(t)$:

<span id="equation-4"></span>

$$
\tilde{\boldsymbol{\omega}}_{\mathrm{S}}(t)
= \boldsymbol{\omega}_{\mathrm{S}}(t)
+ \mathbf{b}_{\omega,\mathrm{S}}
+ \mathbf{n}_{\omega,\mathrm{S}}(t).
\tag{4}
$$

As in the accelerometer model, the bias in [Equation (4)](#equation-4) shifts the readings and noise makes them vary between samples.

For rotation constrained to one fixed axis $i$, let $\theta_i(t)$ be the signed angle in radians and $\omega_i(t)$ the angular velocity about that axis. Their relationship is:

$$
\omega_i(t)=\dot{\theta}_i(t).
$$

Given the initial angle $\theta_i(0)$, integration recovers the angle at time $t$, with $\tau$ denoting integration time:

$$
\theta_i(t)
= \theta_i(0) + \int_0^t \omega_i(\tau)\,\mathrm{d}\tau.
$$

A zero angular velocity means that the sensor is not turning at that moment; its heading still depends on the initial angle and the preceding motion. Using measured angular velocity in the integral also accumulates measurement error. A residual constant bias $b_{\omega,i}$ produces an angle error $b_{\omega,i}t$, while random noise adds further error.

For general three-dimensional motion, the rates of roll, pitch, and yaw are coupled rather than equal to the three angular-velocity components. [Notebook 4](./4-quaternions-and-rotation-representations.ipynb) introduces unit quaternions, which represent orientation and allow it to be updated from angular velocity.

## Gathering IMU measurements from a Duckiedrone

The ROS 2 topic `/mavros/imu/data` carries the IMU readings together with an orientation estimate, which will be explained and tested in more detail in [Notebook 7](./7-interfacing-with-imu.ipynb). Below is a sample message from that topic taken from a stationary, level Duckiedrone. A virtual Duckiedrone publishes the same message type with values produced by the simulator when the Duckiematrix is running and the [virtual Duckiedrone is attached to an entity](https://docs.duckietown.com/ente/duckietown-manual/50-duckiematrix/virtual-duckietown-robots/introduction-to-virtual-duckiebots.html#attaching-virtual-robots).

Although the field is named `linear_acceleration`, the [PX4](https://docs.px4.io/main/)/[MAVROS2](https://github.com/mavlink/mavros) stream used here fills it with measured proper acceleration. Its interpretation therefore depends on the publisher as well as the message definition.

The message uses `frame_id: base_link` for the Duckiedrone body frame, denoted $\mathrm{B}$, whose positive axes point forward ($x$), left ($y$), and upward ($z$) relative to the Duckiedrone. Since these axes can differ from the chip's axes, interpreting the components also requires the sensor-to-body transformation described in [Notebook 5](./5-duckiedrone-imu-architecture.ipynb#from-the-imu-chip-to-px4).

Before reading the message in detail, consider a level, stationary Duckiedrone:

1. What should the three `linear_acceleration` components show, and in what physical units?

2. Predict the three `angular_velocity` components and state their units.

3. Decide whether those stationary readings determine which way the Duckiedrone faces.

The following message was captured on a physical Duckiedrone resting motionless on a bench:

```yaml
header:
  stamp:
    sec: 1788950655
    nanosec: 961939565
  frame_id: base_link
orientation:
  x: 0.0035833670220291654
  y: 0.003122046427906428
  z: -0.7076347443857675
  w: -0.7065624650101054
orientation_covariance:
- 1.0
- 0.0
- 0.0
- 0.0
- 1.0
- 0.0
- 0.0
- 0.0
- 1.0
angular_velocity:
  x: 8.995339157991111e-06
  y: -0.0005893486086279153
  z: 0.000459553673863411
angular_velocity_covariance:
- 1.2184696791468346e-07
- 0.0
- 0.0
- 0.0
- 1.2184696791468346e-07
- 0.0
- 0.0
- 0.0
- 1.2184696791468346e-07
linear_acceleration:
  x: -0.018726181238889694
  y: -0.0861072316765773
  z: 9.91747760772705
linear_acceleration_covariance:
- 8.999999999999999e-08
- 0.0
- 0.0
- 0.0
- 8.999999999999999e-08
- 0.0
- 0.0
- 0.0
- 8.999999999999999e-08
---
```

The message has type `sensor_msgs/msg/Imu`. In the MAVROS topic shown here, `linear_acceleration` and `angular_velocity` contain measured proper acceleration and angular velocity expressed along the Duckiedrone's body axes: forward, left, and up. The sensing elements initially measure along their own sensor axes; PX4 accounts for the sensor and flight-controller mounting orientations, and MAVROS converts PX4's forward-right-down convention to ROS's forward-left-up convention. This interpretation assumes that the configured mounting orientations match the physical installation.

The `orientation` field instead holds an __estimate__ (not a measurement) computed by the flight controller from measurements. Its four values form a quaternion, introduced in [Notebook 4](./4-quaternions-and-rotation-representations.ipynb), while [Notebook 2](./2-imu-estimation-and-drift.ipynb#why-the-imu-data-matters) explains why the flight controller needs this estimate.

For the level, stationary case, the ideal proper acceleration and angular velocity are:

$$
\mathbf{f}_{\mathrm{B}}
=
\begin{bmatrix}
0 \\
0 \\
+9.81
\end{bmatrix}
\,\mathrm{m/s^2},
\qquad
\boldsymbol{\omega}_{\mathrm{B}}
=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
\,\mathrm{rad/s}.
$$

The recorded values differ slightly from these ideal ones; the vertical reading, for example, exceeds the prediction by $9.917-9.81=0.107\,\mathrm{m/s^2}$.

Each group also has a covariance array describing its reported uncertainty, with nine entries forming a three-by-three matrix listed row by row. The diagonal entries describe variances about or along the three axes, but their source depends on the publisher: MAVROS constructs these arrays from configured standard deviations. For example, the acceleration diagonal in this message matches $0.0003^2=9\times10^{-8}\,(\mathrm{m/s^2})^2$, which is consistent with a configured value; estimating noise from the recording still requires multiple samples.

## Summary and next steps

An accelerometer reports proper acceleration along its axes, while a gyroscope reports angular velocity about them. For a level, stationary Duckiedrone with the vertical axis pointing upward, the expected readings are approximately $+9.81\,\mathrm{m/s^2}$ vertically and zero angular velocity.

Since the recorded values differ slightly from these expectations, a sequence of measurements is needed to investigate their mean and variation. [Notebook 2](./2-imu-estimation-and-drift.ipynb) develops that investigation using recorded data, then examines how measurement errors affect estimates obtained by integration.

## Further reading

The [sensor_msgs/Imu message definition](https://github.com/ros2/common_interfaces/blob/jazzy/sensor_msgs/msg/Imu.msg) specifies the message fields, units, and covariance conventions.

The [notation reference](../assets/data/notation.md) collects the symbols used throughout this LX.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()